# C-ADER — Boeing 707 *Château de Maintenon*
## Jumeau numérique — Musée de l'Air et de l'Espace

Ce notebook génère une **expérience interactive de type musée** autour du Boeing 707 conservé au Musée de l'Air et de l'Espace du Bourget.

**Fichiers générés :**
- `style.css` — système de design complet (palette, typographie, animations, responsive)
- `index.html` — page narrative avec viewer 3D, chronologie, fiches techniques
- `script.js` — interactivité, appels API, animations au scroll

**Fichiers requis** (à placer dans le même répertoire) :
- `Boeing_707.glb` — modèle 3D
- `depliant-707.jpg` — archive photographique
- `porte-avant.jpg` — photo zone de mesure
- `logo-mae.png` — logo du musée

In [ ]:
# ============================================================
# CELLULE 1 — Environnement de travail
# ============================================================
import os, sys

# Détection Colab vs local
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.chdir("/content")
    os.makedirs("poc", exist_ok=True)
    os.chdir("poc")
    print("[Colab] Répertoire :", os.getcwd())
    print("Veuillez importer les fichiers requis (Boeing_707.glb, depliant-707.jpg, porte-avant.jpg, logo-mae.png)")
    from google.colab import files
    uploaded = files.upload()
    print("Fichiers importés :", list(uploaded.keys()))
else:
    OUTPUT_DIR = os.path.dirname(os.path.abspath("__file__"))
    os.chdir(OUTPUT_DIR)
    print("[Local] Répertoire :", os.getcwd())

## Feuille de style — `style.css`

Le design repose sur :
- **Palette MAE** étendue avec `--mae-dark-bg` et `--mae-gold` pour les accents
- **Typographie** : Playfair Display (titres) + Inter (corps)
- **Rythme visuel** : alternance sections sombres / claires
- **Animations** : `IntersectionObserver` (`.reveal` → `.revealed`), compteurs, jauges
- **Responsive** : 3 breakpoints (desktop > 980px, tablette 641–980px, mobile < 640px)
- **Accessibilité** : `prefers-reduced-motion`, HTML5 sémantique, `aria`, `<dialog>` natif

In [ ]:
# ============================================================
# CELLULE 2 — Génération de style.css
# ============================================================

css_content = r"""
/* ============================================================
   C-ADER — Design System
   Musée de l'Air et de l'Espace — Boeing 707
   ============================================================ */

/* --- Custom properties --- */
:root {
  --mae-blue:    #0B1D40;
  --mae-red:     #ED1C24;
  --mae-grey:    #798396;
  --mae-light:   #ffffff;
  --mae-border:  rgba(11,29,64,.12);
  --mae-text:    rgba(11,29,64,.85);
  --mae-text2:   rgba(11,29,64,.65);
  --mae-dark-bg: #060E1F;
  --mae-gold:    #C8A96E;
  --radius:      14px;
  --radius-lg:   18px;
}

/* --- Reset & base --- */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html { scroll-behavior: smooth; }

body {
  font-family: 'Inter', system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  background: var(--mae-dark-bg);
  color: var(--mae-text);
  line-height: 1.6;
  overflow-x: hidden;
}

a { color: var(--mae-gold); text-decoration: none; }
a:hover { text-decoration: underline; }

img { max-width: 100%; height: auto; display: block; }

/* --- Typography --- */
h1, h2, h3 { font-family: 'Playfair Display', Georgia, serif; }

/* --- Reveal animations --- */
.reveal {
  opacity: 0;
  transform: translateY(40px);
  transition: opacity .7s ease, transform .7s ease;
}
.reveal.revealed {
  opacity: 1;
  transform: translateY(0);
}
@media (prefers-reduced-motion: reduce) {
  .reveal { opacity: 1; transform: none; transition: none; }
}

/* ============================================================
   NAVBAR
   ============================================================ */
.navbar {
  position: fixed; top: 0; left: 0; width: 100%; z-index: 100;
  padding: 14px 24px;
  display: flex; justify-content: space-between; align-items: center;
  transition: background .4s ease, box-shadow .4s ease;
  background: transparent;
}
.navbar.scrolled {
  background: rgba(6,14,31,.95);
  backdrop-filter: blur(12px);
  box-shadow: 0 2px 20px rgba(0,0,0,.3);
}
.navbar__logo {
  display: flex; align-items: center; gap: 10px;
}
.navbar__logo img { height: 36px; }
.navbar__logo span {
  color: var(--mae-light); font-size: 13px; font-weight: 500;
  line-height: 1.3;
}
.navbar__links {
  display: flex; gap: 24px; list-style: none;
}
.navbar__links a {
  color: rgba(255,255,255,.7); font-size: 13px; font-weight: 500;
  text-transform: uppercase; letter-spacing: .05em;
  transition: color .3s;
}
.navbar__links a:hover { color: var(--mae-gold); text-decoration: none; }

/* ============================================================
   HERO
   ============================================================ */
#hero {
  position: relative; width: 100%; height: 100vh;
  background: var(--mae-dark-bg);
  display: flex; align-items: center; justify-content: center;
}
#hero model-viewer {
  width: 100%; height: 100%;
  --poster-color: transparent;
}
.hero__overlay {
  position: absolute; bottom: 0; left: 0; right: 0;
  padding: 40px 32px 60px;
  background: linear-gradient(transparent, rgba(6,14,31,.85));
  text-align: center; pointer-events: none;
}
.hero__overlay h1 {
  font-size: clamp(2rem, 5vw, 3.6rem);
  color: var(--mae-light); line-height: 1.15;
}
.hero__overlay h1 span { color: var(--mae-gold); }
.hero__overlay p {
  margin-top: 12px; color: rgba(255,255,255,.6);
  font-size: clamp(.9rem, 2vw, 1.1rem);
}
.scroll-indicator {
  position: absolute; bottom: 20px; left: 50%;
  transform: translateX(-50%);
  width: 28px; height: 44px;
  border: 2px solid rgba(255,255,255,.3); border-radius: 14px;
  display: flex; justify-content: center;
}
.scroll-indicator::after {
  content: ''; width: 4px; height: 10px;
  background: var(--mae-gold); border-radius: 2px;
  margin-top: 6px;
  animation: scrollBounce 2s infinite;
}
@keyframes scrollBounce {
  0%, 100% { transform: translateY(0); opacity: 1; }
  50% { transform: translateY(12px); opacity: .3; }
}

/* ============================================================
   SECTIONS — shared
   ============================================================ */
.section { padding: 80px 24px; }
.section--dark { background: var(--mae-dark-bg); color: rgba(255,255,255,.85); }
.section--light { background: #F4F5F7; color: var(--mae-text); }
.section__inner { max-width: 1100px; margin: 0 auto; }
.section__title {
  font-size: clamp(1.6rem, 3vw, 2.4rem);
  margin-bottom: 12px;
}
.section--dark .section__title { color: var(--mae-light); }
.section--light .section__title { color: var(--mae-blue); }
.section__subtitle {
  font-size: .95rem; margin-bottom: 40px;
}
.section--dark .section__subtitle { color: rgba(255,255,255,.5); }
.section--light .section__subtitle { color: var(--mae-text2); }

/* ============================================================
   INTRODUCTION
   ============================================================ */
#introduction .intro-text {
  max-width: 750px; margin: 0 auto;
  font-size: 1.15rem; line-height: 1.9;
  color: rgba(255,255,255,.75);
  text-align: center;
}
#introduction .intro-text strong { color: var(--mae-gold); font-weight: 600; }

/* ============================================================
   HISTOIRE
   ============================================================ */
.histoire-grid {
  display: grid; grid-template-columns: 1fr 1fr;
  gap: 48px; align-items: center;
}
.histoire-text p {
  margin-bottom: 16px; font-size: 1rem; line-height: 1.8;
}
.histoire-image {
  border-radius: var(--radius-lg); overflow: hidden;
  box-shadow: 0 12px 40px rgba(11,29,64,.15);
  cursor: pointer; position: relative;
}
.histoire-image::after {
  content: 'Cliquer pour agrandir';
  position: absolute; bottom: 0; left: 0; right: 0;
  padding: 10px; text-align: center;
  background: rgba(11,29,64,.7); color: #fff;
  font-size: .8rem; opacity: 0;
  transition: opacity .3s;
}
.histoire-image:hover::after { opacity: 1; }

/* ============================================================
   CHRONOLOGIE (Timeline)
   ============================================================ */
.timeline {
  position: relative; padding: 20px 0;
}
.timeline::before {
  content: ''; position: absolute;
  left: 50%; top: 0; bottom: 0; width: 2px;
  background: rgba(200,169,110,.3);
  transform: translateX(-50%);
}
.tl-entry {
  display: flex; width: 100%; margin-bottom: 48px;
  position: relative;
}
.tl-entry:nth-child(odd) { justify-content: flex-start; padding-right: calc(50% + 32px); }
.tl-entry:nth-child(even) { justify-content: flex-end; padding-left: calc(50% + 32px); }
.tl-dot {
  position: absolute; left: 50%; top: 8px;
  width: 14px; height: 14px;
  background: var(--mae-gold); border-radius: 50%;
  transform: translateX(-50%);
  border: 3px solid var(--mae-dark-bg);
  z-index: 2;
}
.tl-card {
  background: rgba(255,255,255,.05);
  border: 1px solid rgba(200,169,110,.2);
  border-radius: var(--radius); padding: 20px 24px;
}
.tl-year {
  font-family: 'Playfair Display', serif;
  font-size: 1.3rem; color: var(--mae-gold);
  margin-bottom: 6px;
}
.tl-card p {
  font-size: .92rem; line-height: 1.7;
  color: rgba(255,255,255,.7);
}

/* ============================================================
   FICHE TECHNIQUE
   ============================================================ */
.specs-grid {
  display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
  gap: 24px;
}
.spec-card {
  background: #fff; border: 1px solid var(--mae-border);
  border-radius: var(--radius); padding: 28px 24px;
  text-align: center;
  box-shadow: 0 6px 24px rgba(11,29,64,.06);
}
.spec-card__icon { font-size: 2rem; margin-bottom: 12px; }
.spec-card__value {
  font-family: 'Playfair Display', serif;
  font-size: 2.2rem; font-weight: 700;
  color: var(--mae-blue);
}
.spec-card__value .unit {
  font-size: .9rem; font-weight: 400;
  color: var(--mae-text2);
}
.spec-card__label {
  margin-top: 6px; font-size: .85rem;
  color: var(--mae-text2); text-transform: uppercase;
  letter-spacing: .04em;
}

/* ============================================================
   WIKIPEDIA
   ============================================================ */
.wiki-card {
  display: flex; gap: 32px; align-items: flex-start;
  background: rgba(255,255,255,.04);
  border: 1px solid rgba(200,169,110,.15);
  border-radius: var(--radius-lg); padding: 32px;
}
.wiki-card img {
  width: 200px; min-width: 200px;
  border-radius: var(--radius);
  object-fit: cover;
}
.wiki-card__body h3 {
  font-size: 1.4rem; color: var(--mae-light);
  margin-bottom: 12px;
}
.wiki-card__body p {
  font-size: .95rem; line-height: 1.8;
  color: rgba(255,255,255,.7);
}
.wiki-card__body a {
  display: inline-block; margin-top: 16px;
  color: var(--mae-gold); font-weight: 500;
}

/* ============================================================
   ENVIRONNEMENT (Weather + AQI)
   ============================================================ */
.env-grid {
  display: grid; grid-template-columns: 1fr 1fr; gap: 32px;
}
.env-card {
  background: rgba(255,255,255,.05);
  border: 1px solid rgba(200,169,110,.15);
  border-radius: var(--radius-lg); padding: 28px;
}
.env-card h3 {
  font-size: 1.2rem; color: var(--mae-light);
  margin-bottom: 20px;
}
.env-row {
  display: flex; justify-content: space-between;
  align-items: center; margin-bottom: 14px;
  font-size: .9rem;
}
.env-row .label { color: rgba(255,255,255,.5); }
.env-row .value { color: var(--mae-light); font-weight: 600; }
.env-weather-icon {
  font-size: 2.8rem; text-align: center;
  margin-bottom: 16px;
}
.env-weather-main {
  text-align: center; margin-bottom: 20px;
}
.env-weather-main .temp {
  font-family: 'Playfair Display', serif;
  font-size: 3rem; color: var(--mae-light);
}
.env-weather-main .desc {
  color: rgba(255,255,255,.6); font-size: .9rem;
}

/* Gauge bars */
.gauge-bar {
  width: 100%; height: 8px;
  background: rgba(255,255,255,.1);
  border-radius: 4px; overflow: hidden;
  margin-top: 6px;
}
.gauge-bar__fill {
  height: 100%; width: 0%; border-radius: 4px;
  transition: width .8s ease;
}
.aqi-badge {
  display: inline-block; padding: 4px 14px;
  border-radius: 20px; font-size: .85rem;
  font-weight: 600; margin-bottom: 16px;
}

/* ============================================================
   MODAL (native dialog)
   ============================================================ */
dialog {
  border: none; border-radius: var(--radius-lg);
  max-width: 90vw; max-height: 90vh;
  padding: 0; overflow: hidden;
  box-shadow: 0 24px 80px rgba(0,0,0,.5);
}
dialog::backdrop { background: rgba(6,14,31,.8); }
.modal-head {
  display: flex; justify-content: space-between;
  align-items: center; padding: 16px 24px;
  border-bottom: 1px solid var(--mae-border);
}
.modal-title { font-family: 'Playfair Display', serif; font-size: 1.1rem; color: var(--mae-blue); }
.modal-subtitle { font-size: .8rem; color: var(--mae-text2); }
.modal-close {
  background: var(--mae-red); color: #fff;
  border: none; border-radius: 8px;
  padding: 8px 16px; cursor: pointer;
  font-size: .85rem; font-weight: 500;
}
.modal-close:hover { opacity: .85; }
.modal-body { padding: 24px; text-align: center; }
.modal-body img { max-height: 70vh; margin: 0 auto; border-radius: var(--radius); }
.modal-caption { margin-top: 12px; font-size: .85rem; color: var(--mae-text2); }

/* ============================================================
   POPOVER
   ============================================================ */
.popover {
  display: none; position: fixed; z-index: 200;
  width: 340px; background: #fff;
  border-radius: var(--radius); overflow: hidden;
  box-shadow: 0 16px 48px rgba(0,0,0,.25);
  pointer-events: none;
}
.popover img { width: 100%; }
.pop-caption {
  padding: 10px 14px; font-size: .8rem;
  color: var(--mae-text2);
}

/* ============================================================
   HOTSPOTS
   ============================================================ */
.hotspot {
  border: 1px solid rgba(255,255,255,.25);
  background: rgba(6,14,31,.7); color: #fff;
  border-radius: 999px; padding: 7px 12px;
  font-size: 12px; cursor: pointer;
  backdrop-filter: blur(6px);
}
.hotspot::before {
  content: ''; display: inline-block;
  width: 8px; height: 8px; border-radius: 50%;
  background: var(--mae-red); margin-right: 8px;
}

/* ============================================================
   FOOTER
   ============================================================ */
.site-footer {
  text-align: center; padding: 40px 24px;
  background: var(--mae-dark-bg);
  border-top: 1px solid rgba(200,169,110,.15);
  color: rgba(255,255,255,.4); font-size: .85rem;
}
.site-footer strong { color: var(--mae-gold); }

/* ============================================================
   RESPONSIVE
   ============================================================ */
@media (max-width: 980px) {
  .histoire-grid { grid-template-columns: 1fr; }
  .env-grid { grid-template-columns: 1fr; }
  .wiki-card { flex-direction: column; }
  .wiki-card img { width: 100%; min-width: auto; }
  .navbar__links { gap: 14px; }
  .tl-entry:nth-child(odd),
  .tl-entry:nth-child(even) {
    padding-left: 40px; padding-right: 0;
    justify-content: flex-start;
  }
  .timeline::before { left: 12px; }
  .tl-dot { left: 12px; }
}
@media (max-width: 640px) {
  .section { padding: 48px 16px; }
  .navbar__links { display: none; }
  .specs-grid { grid-template-columns: 1fr 1fr; gap: 14px; }
  .spec-card { padding: 18px 14px; }
  .spec-card__value { font-size: 1.6rem; }
  #hero { height: 80vh; }
}
"""

with open("style.css", "w", encoding="utf-8") as f:
    f.write(css_content.strip())

print("style.css généré (", len(css_content.strip().splitlines()), "lignes )")

## Page HTML — `index.html`

Structure narrative en 9 sections :
1. **Navbar** — navigation fixe, transparente sur le hero
2. **Hero** — viewer 3D plein écran avec titre en surimpression
3. **Introduction** — texte cinématique
4. **Histoire** — texte + image d'archive avec modale
5. **Chronologie** — timeline verticale interactive (10 entrées)
6. **Fiche technique** — cartes animées avec compteurs
7. **Wikipedia** — extrait live de l'API Wikipédia FR
8. **Environnement** — météo + qualité de l'air (Open-Meteo)
9. **Footer** — crédits

In [ ]:
# ============================================================
# CELLULE 3 — Génération de index.html
# ============================================================

html_content = r"""
<!doctype html>
<html lang="fr">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>C-ADER — Boeing 707 Château de Maintenon | Musee de l'Air et de l'Espace</title>
  <meta name="description" content="Jumeau numerique du Boeing 707 Château de Maintenon — Musee de l'Air et de l'Espace du Bourget">

  <link rel="stylesheet" href="style.css">
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Playfair+Display:wght@400;600;700&display=swap" rel="stylesheet">
  <script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
</head>
<body>

  <!-- Popover -->
  <div class="popover" id="hoverPopover" aria-hidden="true">
    <img id="popImg" src="" alt="">
    <div class="pop-caption" id="popCaption"></div>
  </div>

  <!-- Modal -->
  <dialog id="imgModal" aria-label="Visionneuse d'image">
    <div class="modal-head">
      <div>
        <div id="imgTitle" class="modal-title">Archive</div>
        <div class="modal-subtitle">Musee de l'Air et de l'Espace</div>
      </div>
      <button id="closeImgModal" class="modal-close" aria-label="Fermer">Fermer</button>
    </div>
    <div class="modal-body">
      <img id="imgViewer" src="" alt="Image agrandie">
      <p id="imgCaption" class="modal-caption"></p>
    </div>
  </dialog>

  <!-- ======== NAVBAR ======== -->
  <nav class="navbar" id="navbar" aria-label="Navigation principale">
    <div class="navbar__logo">
      <img src="logo-mae.png" alt="Logo Musee de l'Air et de l'Espace">
      <span>Musee de l'Air<br>et de l'Espace</span>
    </div>
    <ul class="navbar__links">
      <li><a href="#introduction">Introduction</a></li>
      <li><a href="#histoire">Histoire</a></li>
      <li><a href="#chronologie">Chronologie</a></li>
      <li><a href="#fiche-technique">Fiche technique</a></li>
      <li><a href="#environnement">Environnement</a></li>
    </ul>
  </nav>

  <!-- ======== HERO ======== -->
  <section id="hero">
    <model-viewer
      src="Boeing_707.glb"
      camera-controls touch-action="pan-y"
      tone-mapping="neutral"
      shadow-intensity="0.6"
      exposure="1.1"
      environment-image="neutral"
      alt="Modele 3D — Boeing 707 Air France"
      style="background-color: #060E1F;">

      <button class="hotspot" slot="hotspot-porte"
              data-position="1.2 1.4 2.1" data-normal="0 1 0"
              data-src="porte-avant.jpg"
              data-title="Zone de mesure XRF — Porte avant"
              data-caption="Analyse de corrosion par fluorescence X (XRF).">
        Porte avant
      </button>
    </model-viewer>

    <div class="hero__overlay">
      <h1>Boeing 707 <span>Château de Maintenon</span></h1>
      <p>Jumeau numerique — Musee de l'Air et de l'Espace du Bourget</p>
    </div>
    <div class="scroll-indicator" aria-hidden="true"></div>
  </section>

  <!-- ======== INTRODUCTION ======== -->
  <section id="introduction" class="section section--dark">
    <div class="section__inner reveal">
      <div class="intro-text">
        <p>
          Au coeur du <strong>Musee de l'Air et de l'Espace</strong> du Bourget,
          le Boeing 707-328 <strong>F-BHSB</strong>, baptise
          <strong>Château de Maintenon</strong>, temoigne d'une epoque
          ou l'aviation commerciale a change le monde.
          Livre a Air France en 1959, cet appareil a relie Paris aux
          grandes capitales avant de devenir piece de musee en 1986.
        </p>
        <p style="margin-top: 20px;">
          Ce jumeau numerique propose une exploration immersive de sa structure,
          de son histoire et de son environnement de conservation,
          au service de la preservation du patrimoine aeronautique.
        </p>
      </div>
    </div>
  </section>

  <!-- ======== HISTOIRE ======== -->
  <section id="histoire" class="section section--light">
    <div class="section__inner">
      <h2 class="section__title reveal">Le Boeing 707 et Air France</h2>
      <p class="section__subtitle reveal">Un avion qui a democratise le voyage transatlantique</p>
      <div class="histoire-grid">
        <div class="histoire-text reveal">
          <p>
            Le Boeing 707 marque un tournant dans l'histoire de l'aviation civile.
            Premier avion de ligne a reaction largement produit, il a permis de
            reduire de moitie le temps de traversee de l'Atlantique, faisant
            entrer le transport aerien dans l'ere moderne.
          </p>
          <p>
            Air France exploita 36 exemplaires, chacun portant le nom d'un château
            de France — la fameuse serie des <em>Châteaux in the Sky</em>.
            Le F-BHSB, <strong>Château de Maintenon</strong>, fut l'un des premiers
            livres a la compagnie.
          </p>
          <p>
            Apres 27 ans de service et plus de 40 000 heures de vol,
            l'appareil fut confie au Musee de l'Air et de l'Espace,
            ou il est conserve sur le tarmac historique du Bourget.
          </p>
          <p>
            <a href="#"
               class="open-archive"
               data-src="depliant-707.jpg"
               data-title="Depliant promotionnel Air France — Boeing 707"
               data-caption="Archive de la serie Châteaux in the Sky — collection Musee de l'Air et de l'Espace.">
              Voir le depliant d'epoque &rarr;
            </a>
          </p>
        </div>
        <div class="histoire-image reveal open-archive"
             data-src="depliant-707.jpg"
             data-title="Depliant promotionnel Air France — Boeing 707"
             data-caption="Archive de la serie Châteaux in the Sky — collection Musee de l'Air et de l'Espace."
             role="button" tabindex="0" aria-label="Agrandir le depliant">
          <img src="depliant-707.jpg" alt="Depliant Boeing 707 Air France" loading="lazy">
        </div>
      </div>
    </div>
  </section>

  <!-- ======== CHRONOLOGIE ======== -->
  <section id="chronologie" class="section section--dark">
    <div class="section__inner">
      <h2 class="section__title reveal">Chronologie</h2>
      <p class="section__subtitle reveal">50 ans d'histoire du Boeing 707</p>
      <div class="timeline">

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1954</div>
            <p>Premier vol du prototype Boeing 367-80 (Dash 80), ancetre direct du 707, le 15 juillet a Renton (Washington).</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1957</div>
            <p>Certification FAA du Boeing 707-120. Pan American World Airways passe la premiere commande commerciale.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1958</div>
            <p>26 octobre : premier vol commercial du 707 (Pan Am, New York — Paris). L'ere du jet commence.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1959</div>
            <p>Air France recoit ses premiers Boeing 707-328 Intercontinental et lance les Châteaux in the Sky sur les lignes transatlantiques.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1960</div>
            <p>Le F-BHSB Château de Maintenon entre en service. Il assure les liaisons Paris — New York, Montreal et Amerique du Sud.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1962</div>
            <p>Boeing lance le 707-320B, version amelioree avec reacteurs JT3D turbofan, reduisant consommation et bruit.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1969</div>
            <p>Le Boeing 747 effectue son premier vol. Le 707, bien que depasse en capacite, reste en service sur de nombreuses lignes.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1979</div>
            <p>Fin de la production du Boeing 707 apres 1 010 exemplaires civils construits. Il a transforme l'aviation mondiale.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">1986</div>
            <p>Le Château de Maintenon est retire du service et confie au Musee de l'Air et de l'Espace du Bourget pour preservation.</p>
          </div>
        </div>

        <div class="tl-entry reveal"><div class="tl-dot"></div>
          <div class="tl-card">
            <div class="tl-year">2003</div>
            <p>Lancement des premiers projets de numerisation du patrimoine aeronautique au Bourget, prefigurant les jumeaux numeriques.</p>
          </div>
        </div>

      </div>
    </div>
  </section>

  <!-- ======== FICHE TECHNIQUE ======== -->
  <section id="fiche-technique" class="section section--light">
    <div class="section__inner">
      <h2 class="section__title reveal">Fiche technique</h2>
      <p class="section__subtitle reveal">Boeing 707-328 Intercontinental</p>
      <div class="specs-grid">
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#9992;</div>
          <div class="spec-card__value"><span data-count="44.04">0</span> <span class="unit">m</span></div>
          <div class="spec-card__label">Envergure</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#128207;</div>
          <div class="spec-card__value"><span data-count="46.61">0</span> <span class="unit">m</span></div>
          <div class="spec-card__label">Longueur</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#9650;</div>
          <div class="spec-card__value"><span data-count="12.93">0</span> <span class="unit">m</span></div>
          <div class="spec-card__label">Hauteur</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#128168;</div>
          <div class="spec-card__value"><span data-count="977">0</span> <span class="unit">km/h</span></div>
          <div class="spec-card__label">Vitesse de croisiere</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#127757;</div>
          <div class="spec-card__value"><span data-count="7400">0</span> <span class="unit">km</span></div>
          <div class="spec-card__label">Autonomie</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#128101;</div>
          <div class="spec-card__value"><span data-count="189">0</span></div>
          <div class="spec-card__label">Passagers max</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#9881;</div>
          <div class="spec-card__value"><span data-count="4">0</span> <span class="unit">x P&amp;W JT4A</span></div>
          <div class="spec-card__label">Moteurs</div>
        </div>
        <div class="spec-card reveal">
          <div class="spec-card__icon" aria-hidden="true">&#9878;</div>
          <div class="spec-card__value"><span data-count="151">0</span> <span class="unit">t</span></div>
          <div class="spec-card__label">Masse max au decollage</div>
        </div>
      </div>
    </div>
  </section>

  <!-- ======== WIKIPEDIA ======== -->
  <section id="wikipedia" class="section section--dark">
    <div class="section__inner">
      <h2 class="section__title reveal">Wikipedia</h2>
      <p class="section__subtitle reveal">Extrait en direct de l'encyclopedie libre</p>
      <div class="wiki-card reveal" id="wikiCard">
        <img id="wikiImg" src="" alt="Boeing 707">
        <div class="wiki-card__body">
          <h3 id="wikiTitle">Boeing 707</h3>
          <p id="wikiExtract">Chargement...</p>
          <a id="wikiLink" href="https://fr.wikipedia.org/wiki/Boeing_707" target="_blank" rel="noopener">Lire l'article complet &rarr;</a>
        </div>
      </div>
    </div>
  </section>

  <!-- ======== ENVIRONNEMENT ======== -->
  <section id="environnement" class="section section--dark">
    <div class="section__inner">
      <h2 class="section__title reveal">Conditions environnementales</h2>
      <p class="section__subtitle reveal">Donnees en temps reel — Aeroport du Bourget (48.95°N, 2.43°E)</p>
      <div class="env-grid">

        <!-- Weather card -->
        <div class="env-card reveal" id="weatherCard">
          <h3>Meteo</h3>
          <div class="env-weather-icon" id="weatherIcon">--</div>
          <div class="env-weather-main">
            <div class="temp" id="weatherTemp">--</div>
            <div class="desc" id="weatherDesc">Chargement...</div>
          </div>
          <div class="env-row"><span class="label">Humidite</span><span class="value" id="weatherHum">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="humGauge"></div></div>
          <div class="env-row"><span class="label">Point de rosee</span><span class="value" id="weatherDew">--</span></div>
          <div class="env-row"><span class="label">Vent</span><span class="value" id="weatherWind">--</span></div>
        </div>

        <!-- AQI card -->
        <div class="env-card reveal" id="aqiCard">
          <h3>Qualite de l'air</h3>
          <div id="aqiBadge" class="aqi-badge">--</div>
          <div class="env-row"><span class="label">IQA europeen</span><span class="value" id="aqiValue">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="aqiGauge"></div></div>
          <div class="env-row"><span class="label">PM10</span><span class="value" id="aqiPM10">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="pm10Gauge"></div></div>
          <div class="env-row"><span class="label">PM2.5</span><span class="value" id="aqiPM25">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="pm25Gauge"></div></div>
          <div class="env-row"><span class="label">NO₂</span><span class="value" id="aqiNO2">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="no2Gauge"></div></div>
          <div class="env-row"><span class="label">O₃</span><span class="value" id="aqiO3">--</span></div>
          <div class="gauge-bar"><div class="gauge-bar__fill" id="o3Gauge"></div></div>
        </div>

      </div>
    </div>
  </section>

  <!-- ======== FOOTER ======== -->
  <footer class="site-footer">
    <p><strong>C-ADER</strong> — Jumeau numerique du Boeing 707</p>
    <p>Musee de l'Air et de l'Espace &middot; Le Bourget</p>
  </footer>

  <script src="script.js"></script>
</body>
</html>
"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content.strip())

print("index.html genere (", len(html_content.strip().splitlines()), "lignes )")

## JavaScript — `script.js`

Toute l'interactivite de la page :
- **Navbar** : transparente sur le hero, opaque au scroll
- **Scroll reveal** : `IntersectionObserver` sur les elements `.reveal`
- **Popover + Modal** : survol et clic sur les archives / hotspots
- **Compteurs animes** : chiffres de la fiche technique
- **3 appels API** : Weather, Air Quality, Wikipedia (tous avec fallback francais)

In [ ]:
# ============================================================
# CELLULE 4 — Génération de script.js
# ============================================================

js_content = r"""
/* ============================================================
   C-ADER — Boeing 707 — script.js
   Musee de l'Air et de l'Espace
   ============================================================ */

(() => {
  'use strict';

  /* ----------------------------------------------------------
     1. NAVBAR — transparent ↔ solid on scroll
     ---------------------------------------------------------- */
  const navbar = document.getElementById('navbar');
  const heroSection = document.getElementById('hero');

  function updateNavbar() {
    if (!navbar || !heroSection) return;
    const threshold = heroSection.offsetHeight - 80;
    navbar.classList.toggle('scrolled', window.scrollY > threshold);
  }
  window.addEventListener('scroll', updateNavbar, { passive: true });
  updateNavbar();

  /* ----------------------------------------------------------
     2. SCROLL REVEAL — IntersectionObserver
     ---------------------------------------------------------- */
  const prefersReduced = window.matchMedia('(prefers-reduced-motion: reduce)').matches;

  if (!prefersReduced) {
    const revealObserver = new IntersectionObserver((entries) => {
      entries.forEach(entry => {
        if (entry.isIntersecting) {
          entry.target.classList.add('revealed');
          revealObserver.unobserve(entry.target);
        }
      });
    }, { threshold: 0.12 });

    document.querySelectorAll('.reveal').forEach(el => revealObserver.observe(el));
  } else {
    document.querySelectorAll('.reveal').forEach(el => el.classList.add('revealed'));
  }

  /* ----------------------------------------------------------
     3. POPOVER + MODAL
     ---------------------------------------------------------- */
  const pop = document.getElementById('hoverPopover');
  const popImg = document.getElementById('popImg');
  const popCaption = document.getElementById('popCaption');

  const imgModal = document.getElementById('imgModal');
  const imgViewer = document.getElementById('imgViewer');
  const imgTitle = document.getElementById('imgTitle');
  const imgCaption = document.getElementById('imgCaption');

  document.getElementById('closeImgModal').addEventListener('click', () => imgModal.close());

  function movePopover(e) {
    const w = pop.offsetWidth || 340;
    const h = pop.offsetHeight || 260;
    let x = e.clientX + 16;
    let y = e.clientY + 16;
    if (x + w > window.innerWidth) x = e.clientX - w - 16;
    if (y + h > window.innerHeight) y = e.clientY - h - 16;
    pop.style.left = x + 'px';
    pop.style.top = y + 'px';
  }

  function showPopover(el, e) {
    popImg.src = el.dataset.src;
    popCaption.textContent = el.dataset.caption || '';
    pop.style.display = 'block';
    movePopover(e);
  }

  function hidePopover() {
    pop.style.display = 'none';
    popImg.src = '';
  }

  function openModal(el) {
    imgViewer.src = el.dataset.src;
    imgTitle.textContent = el.dataset.title || 'Archive';
    imgCaption.textContent = el.dataset.caption || '';
    imgModal.showModal();
  }

  // Hotspots
  document.querySelectorAll('.hotspot').forEach(h => {
    h.addEventListener('mouseenter', e => showPopover(h, e));
    h.addEventListener('mousemove', e => movePopover(e));
    h.addEventListener('mouseleave', hidePopover);
    h.addEventListener('click', () => openModal(h));
  });

  // Archive links and image containers
  document.querySelectorAll('.open-archive').forEach(link => {
    if (link.tagName === 'A') {
      link.addEventListener('mouseenter', e => showPopover(link, e));
      link.addEventListener('mousemove', e => movePopover(e));
      link.addEventListener('mouseleave', hidePopover);
    }
    link.addEventListener('click', e => {
      e.preventDefault();
      openModal(link);
    });
    // Keyboard accessibility
    link.addEventListener('keydown', e => {
      if (e.key === 'Enter' || e.key === ' ') {
        e.preventDefault();
        openModal(link);
      }
    });
  });

  window.addEventListener('scroll', hidePopover, { passive: true });

  /* ----------------------------------------------------------
     4. COUNTER ANIMATION — spec cards
     ---------------------------------------------------------- */
  function animateCounter(el, target, duration) {
    const isFloat = String(target).includes('.');
    const start = performance.now();

    function step(now) {
      const progress = Math.min((now - start) / duration, 1);
      const eased = 1 - Math.pow(1 - progress, 3); // ease-out cubic
      const current = eased * target;
      el.textContent = isFloat ? current.toFixed(2) : Math.round(current);
      if (progress < 1) requestAnimationFrame(step);
    }
    requestAnimationFrame(step);
  }

  const counterObserver = new IntersectionObserver((entries) => {
    entries.forEach(entry => {
      if (entry.isIntersecting) {
        const target = parseFloat(entry.target.dataset.count);
        animateCounter(entry.target, target, 1600);
        counterObserver.unobserve(entry.target);
      }
    });
  }, { threshold: 0.5 });

  document.querySelectorAll('[data-count]').forEach(el => counterObserver.observe(el));

  /* ----------------------------------------------------------
     5. WEATHER API — Open-Meteo
     ---------------------------------------------------------- */
  const WMO_LABELS = {
    0: ['Ciel degage', '☀️'], 1: ['Peu nuageux', '🌤️'],
    2: ['Partiellement nuageux', '⛅'], 3: ['Couvert', '☁️'],
    45: ['Brouillard', '🌫️'], 48: ['Brouillard givrant', '🌫️'],
    51: ['Bruine legere', '🌦️'], 53: ['Bruine', '🌦️'], 55: ['Bruine dense', '🌧️'],
    61: ['Pluie legere', '🌦️'], 63: ['Pluie', '🌧️'], 65: ['Pluie forte', '🌧️'],
    71: ['Neige legere', '🌨️'], 73: ['Neige', '❄️'], 75: ['Neige forte', '❄️'],
    80: ['Averses', '🌦️'], 81: ['Averses moderes', '🌧️'], 82: ['Averses violentes', '⛈️'],
    95: ['Orage', '⛈️'], 96: ['Orage / grele', '⛈️'], 99: ['Orage / forte grele', '⛈️']
  };

  async function loadWeather() {
    try {
      const url = 'https://api.open-meteo.com/v1/forecast?latitude=48.953&longitude=2.430&current=temperature_2m,relative_humidity_2m,dew_point_2m,weather_code,wind_speed_10m';
      const res = await fetch(url);
      const data = await res.json();
      const c = data.current;

      const wmo = WMO_LABELS[c.weather_code] || ['--', '🌡️'];
      document.getElementById('weatherIcon').textContent = wmo[1];
      document.getElementById('weatherDesc').textContent = wmo[0];
      document.getElementById('weatherTemp').textContent = c.temperature_2m + ' °C';
      document.getElementById('weatherHum').textContent = c.relative_humidity_2m + ' %';
      document.getElementById('weatherDew').textContent = c.dew_point_2m + ' °C';
      document.getElementById('weatherWind').textContent = c.wind_speed_10m + ' km/h';

      // Humidity gauge
      const humGauge = document.getElementById('humGauge');
      humGauge.style.width = c.relative_humidity_2m + '%';
      if (c.relative_humidity_2m < 60) humGauge.style.background = '#4caf50';
      else if (c.relative_humidity_2m < 80) humGauge.style.background = '#ff9800';
      else humGauge.style.background = '#f44336';
    } catch {
      document.getElementById('weatherDesc').textContent = 'Donnees indisponibles';
    }
  }

  /* ----------------------------------------------------------
     6. AIR QUALITY API — Open-Meteo
     ---------------------------------------------------------- */
  function aqiColor(val) {
    if (val <= 20) return { bg: '#4caf50', label: 'Bon' };
    if (val <= 40) return { bg: '#8bc34a', label: 'Correct' };
    if (val <= 60) return { bg: '#ff9800', label: 'Moyen' };
    if (val <= 80) return { bg: '#ff5722', label: 'Mediocre' };
    if (val <= 100) return { bg: '#f44336', label: 'Mauvais' };
    return { bg: '#9c27b0', label: 'Tres mauvais' };
  }

  function setGauge(id, value, max) {
    const el = document.getElementById(id);
    if (!el) return;
    const pct = Math.min(100, (value / max) * 100);
    el.style.width = pct + '%';
    el.style.background = aqiColor(pct).bg;
  }

  async function loadAirQuality() {
    try {
      const url = 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=48.953&longitude=2.430&current=european_aqi,pm10,pm2_5,nitrogen_dioxide,ozone';
      const res = await fetch(url);
      const data = await res.json();
      const c = data.current;

      const aqi = c.european_aqi;
      const info = aqiColor(aqi);
      const badge = document.getElementById('aqiBadge');
      badge.textContent = info.label;
      badge.style.background = info.bg;
      badge.style.color = '#fff';

      document.getElementById('aqiValue').textContent = aqi;
      setGauge('aqiGauge', aqi, 100);

      document.getElementById('aqiPM10').textContent = c.pm10 + ' µg/m³';
      setGauge('pm10Gauge', c.pm10, 80);

      document.getElementById('aqiPM25').textContent = c.pm2_5 + ' µg/m³';
      setGauge('pm25Gauge', c.pm2_5, 50);

      document.getElementById('aqiNO2').textContent = c.nitrogen_dioxide + ' µg/m³';
      setGauge('no2Gauge', c.nitrogen_dioxide, 200);

      document.getElementById('aqiO3').textContent = c.ozone + ' µg/m³';
      setGauge('o3Gauge', c.ozone, 180);
    } catch {
      document.getElementById('aqiBadge').textContent = 'Donnees indisponibles';
    }
  }

  /* ----------------------------------------------------------
     7. WIKIPEDIA API — French extract
     ---------------------------------------------------------- */
  async function loadWikipedia() {
    try {
      const url = 'https://fr.wikipedia.org/api/rest_v1/page/summary/Boeing_707';
      const res = await fetch(url);
      const data = await res.json();

      document.getElementById('wikiTitle').textContent = data.title || 'Boeing 707';
      document.getElementById('wikiExtract').textContent = data.extract || 'Donnees indisponibles.';

      if (data.thumbnail && data.thumbnail.source) {
        document.getElementById('wikiImg').src = data.thumbnail.source;
        document.getElementById('wikiImg').alt = data.title;
      }
      if (data.content_urls && data.content_urls.desktop) {
        document.getElementById('wikiLink').href = data.content_urls.desktop.page;
      }
    } catch {
      document.getElementById('wikiExtract').textContent = 'Donnees indisponibles.';
    }
  }

  /* ----------------------------------------------------------
     8. INIT
     ---------------------------------------------------------- */
  loadWeather();
  loadAirQuality();
  loadWikipedia();

})();
"""

with open("script.js", "w", encoding="utf-8") as f:
    f.write(js_content.strip())

print("script.js genere (", len(js_content.strip().splitlines()), "lignes )")

In [ ]:
# ============================================================
# CELLULE 5 — Vérification des assets
# ============================================================
import os

required = ['Boeing_707.glb', 'depliant-707.jpg', 'porte-avant.jpg', 'logo-mae.png']
generated = ['style.css', 'index.html', 'script.js']

print('=== Fichiers generés ===')
for f in generated:
    status = 'OK' if os.path.isfile(f) else 'MANQUANT'
    size = os.path.getsize(f) if os.path.isfile(f) else 0
    print(f'  {f:20s} {status:10s} ({size:,} octets)')

print('\n=== Assets requis ===')
for f in required:
    status = 'OK' if os.path.isfile(f) else 'MANQUANT'
    size = os.path.getsize(f) if os.path.isfile(f) else 0
    print(f'  {f:20s} {status:10s} ({size:,} octets)')

missing = [f for f in required if not os.path.isfile(f)]
if missing:
    print(f'\n⚠ Assets manquants : {missing}')
else:
    print('\nTous les fichiers sont presents.')

In [ ]:
# ============================================================
# CELLULE 6 — Lancement du serveur HTTP + prévisualisation
# ============================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.Popen(['python3', '-m', 'http.server', '8000'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, height=900, path='/')
else:
    import webbrowser, threading, http.server, functools

    PORT = 8000
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=os.getcwd())
    server = http.server.HTTPServer(('', PORT), handler)
    t = threading.Thread(target=server.serve_forever, daemon=True)
    t.start()

    url = f'http://localhost:{PORT}/index.html'
    print(f'Serveur demarre : {url}')
    webbrowser.open(url)